In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models
import tensorflow_datasets as tfds
import numpy as np
from sklearn.metrics import classification_report

(ds_train, ds_test), ds_info = tfds.load(
    'tf_flowers',
    split=['train[:70%]', 'train[70%:]'],
    as_supervised=True,
    with_info=True
)

NUM_CLASSES = ds_info.features['label'].num_classes
print("Classes:", NUM_CLASSES)

IMG_SIZE = 224
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = image / 255.0
    return image, label

ds_train = ds_train.map(preprocess).shuffle(1000).batch(BATCH_SIZE)
ds_test = ds_test.map(preprocess).batch(BATCH_SIZE)

data_augmentation = tf.keras.Sequential([
    layers.RandomFlip("horizontal"),
    layers.RandomRotation(0.1),
    layers.RandomBrightness(0.2),
    layers.RandomZoom(0.1)
])





Dl Completed...: 0 url [00:00, ? url/s]

Dl Size...: 0 MiB [00:00, ? MiB/s]

Generating splits...:   0%|          | 0/1 [00:00<?, ? splits/s]

Generating train examples...: 0 examples [00:00, ? examples/s]

Shuffling /root/tensorflow_datasets/tf_flowers/incomplete.SI6K7I_3.0.1/tf_flowers-train.tfrecord*...:   0%|   …

Dataset tf_flowers downloaded and prepared to /root/tensorflow_datasets/tf_flowers/3.0.1. Subsequent calls will reuse this data.
Classes: 5


In [6]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)

base_model.trainable = False  # Phase 1

model = models.Sequential([
    data_augmentation,
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(NUM_CLASSES, activation='softmax')
])

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history1 = model.fit(
    ds_train,
    epochs=5,
    validation_data=ds_test
)
base_model.trainable = True

for layer in base_model.layers[:-30]:
    layer.trainable = False

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history2 = model.fit(
    ds_train,
    epochs=3,
    validation_data=ds_test
)

9406464/9406464 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
Epoch 1/5
81/81 ━━━━━━━━━━━━━━━━━━━━ 185s 2s/step - accuracy: 0.2701 - loss: 1.6396 - val_accuracy: 0.5186 - val_loss: 1.2543
Epoch 2/5
81/81 ━━━━━━━━━━━━━━━━━━━━ 164s 2s/step - accuracy: 0.3134 - loss: 1.5014 - val_accuracy: 0.6258 - val_loss: 1.0281
Epoch 3/5
81/81 ━━━━━━━━━━━━━━━━━━━━ 198s 2s/step - accuracy: 0.3235 - loss: 1.4651 - val_accuracy: 0.6585 - val_loss: 0.9729
Epoch 4/5
81/81 ━━━━━━━━━━━━━━━━━━━━ 209s 2s/step - accuracy: 0.3593 - loss: 1.4477 - val_accuracy: 0.6785 - val_loss: 0.8621
Epoch 5/5
81/81 ━━━━━━━━━━━━━━━━━━━━ 163s 2s/step - accuracy: 0.3725 - loss: 1.4364 - val_accuracy: 0.6830 - val_loss: 0.8297
Epoch 1/3
81/81 ━━━━━━━━━━━━━━━━━━━━ 252s 3s/step - accuracy: 0.2993 - loss: 1.5544 - val_accuracy: 0.6930 - val_loss: 0.8057
Epoch 2/3
81/81 ━━━━━━━━━━━━━━━━━━━━ 206s 3s/step - accuracy: 0.3429 - loss: 1.4595 - val_accuracy: 0.6966 - val_loss: 0.7882
Epoch 3/3
81/81 ━━━━━━━━━━━━━━━━━━━━ 194s 2s/step - accuracy: 0.3453 

In [7]:
y_true = []
y_pred = []

for images, labels in ds_test:
    preds = model.predict(images)
    y_true.extend(labels.numpy())
    y_pred.extend(np.argmax(preds, axis=1))

print(classification_report(y_true, y_pred))

1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 2s 2s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 1s/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 1s 

In [8]:
import tensorflow as tf

converter = tf.lite.TFLiteConverter.from_keras_model(model)
tflite_model = converter.convert()

with open("smartbin_model.tflite", "wb") as f:
    f.write(tflite_model)

Saved artifact at '/tmp/tmp266jzsfm'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 224, 224, 3), dtype=tf.float32, name='keras_tensor_154')
Output Type:
  TensorSpec(shape=(None, 5), dtype=tf.float32, name=None)
Captures:
  136642733893648: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136642733899408: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136642733896528: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136642733899216: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136642733898640: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136642733899600: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136642733131536: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136642733131728: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136642733130960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  136642733130384: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1366427331